In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost lightgbm joblib

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 115.7 kB/s eta 0:00:09
   -------------- -------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
"""
TS-12 Early Academic Risk Detection
====================================
ML Pipeline: Risk Score Prediction (Regression)
+ Rule-based Risk Label from predicted score
 
Features : attendance, marks, assignment, lms
Target   : risk_score (continuous)
Labels   : Low / Medium / High (rule-based on score)
"""
 
# ─────────────────────────────────────────────
# 0. IMPORTS
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os
 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
 
# XGBoost & LightGBM — install if needed:
# pip install xgboost lightgbm
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not installed. Run: pip install xgboost")
 
try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("⚠️  LightGBM not installed. Run: pip install lightgbm")
 
warnings.filterwarnings("ignore")
os.makedirs("outputs", exist_ok=True)
 

In [3]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: Loading Data")
print("=" * 60)
 
CSV_PATH = "TS-PS12.csv"          # ← Change to your actual path
df = pd.read_csv(CSV_PATH)
 
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nNull Values:\n{df.isnull().sum()}")
print(f"\nBasic Stats:\n{df.describe()}")

STEP 1: Loading Data
Shape: (50000, 7)

First 5 rows:
   student_id  attendance  marks  assignment  lms  risk_score risk_label
0           1          84     96          34   20        28.8        Low
1           2          87     72          87   28        23.4        Low
2           3          93     87          95   26        15.1        Low
3           4          40     66          83   52        42.4     Medium
4           5          43     99          50   75        35.6     Medium

Data Types:
student_id      int64
attendance      int64
marks           int64
assignment      int64
lms             int64
risk_score    float64
risk_label     object
dtype: object

Null Values:
student_id    0
attendance    0
marks         0
assignment    0
lms           0
risk_score    0
risk_label    0
dtype: int64

Basic Stats:
         student_id    attendance         marks    assignment           lms  \
count  50000.000000  50000.000000  50000.000000  50000.000000  50000.000000   
mean   25000.500

In [4]:
 
# ─────────────────────────────────────────────
# 2. EDA — EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: Exploratory Data Analysis")
print("=" * 60)
 
# Distribution of risk_score
plt.figure(figsize=(14, 4))
 
plt.subplot(1, 3, 1)
df["risk_score"].hist(bins=40, color="steelblue", edgecolor="white")
plt.title("Risk Score Distribution")
plt.xlabel("risk_score")
 
plt.subplot(1, 3, 2)
df["risk_label"].value_counts().plot(kind="bar", color=["green", "orange", "red"])
plt.title("Risk Label Counts")
plt.xticks(rotation=0)
 
plt.subplot(1, 3, 3)
score_by_label = df.groupby("risk_label")["risk_score"].mean().reindex(["Low", "Medium", "High"])
score_by_label.plot(kind="bar", color=["green", "orange", "red"])
plt.title("Avg Risk Score by Label")
plt.xticks(rotation=0)
 
plt.tight_layout()
plt.savefig("outputs/eda_distributions.png", dpi=150)
plt.close()
print("✅ Saved: outputs/eda_distributions.png")
 
# Correlation heatmap
FEATURES = ["attendance", "marks", "assignment", "lms"]
plt.figure(figsize=(7, 5))
corr = df[FEATURES + ["risk_score"]].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Feature Correlation with Risk Score")
plt.tight_layout()
plt.savefig("outputs/eda_correlation.png", dpi=150)
plt.close()
print("✅ Saved: outputs/eda_correlation.png")
 
# Print correlation with target
print("\nCorrelation of features with risk_score:")
print(corr["risk_score"].sort_values())
 
# Derive thresholds from actual data
low_max  = df[df["risk_label"] == "Low"]["risk_score"].max()
low_min  = df[df["risk_label"] == "Low"]["risk_score"].min()
med_max  = df[df["risk_label"] == "Medium"]["risk_score"].max()
med_min  = df[df["risk_label"] == "Medium"]["risk_score"].min()
high_min = df[df["risk_label"] == "High"]["risk_score"].min()
print(f"\nRisk Score Ranges in data:")
print(f"  Low    : {low_min:.1f} – {low_max:.1f}")
print(f"  Medium : {med_min:.1f} – {med_max:.1f}")
print(f"  High   : {high_min:.1f}+")
 
 


STEP 2: Exploratory Data Analysis
✅ Saved: outputs/eda_distributions.png
✅ Saved: outputs/eda_correlation.png

Correlation of features with risk_score:
attendance   -0.671528
marks        -0.589910
assignment   -0.393389
lms          -0.224985
risk_score    1.000000
Name: risk_score, dtype: float64

Risk Score Ranges in data:
  Low    : 3.2 – 30.0
  Medium : 30.0 – 60.0
  High   : 60.0+


In [5]:
 
# ─────────────────────────────────────────────
# 3. PREPROCESSING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: Preprocessing")
print("=" * 60)
 
# Drop nulls if any
df.dropna(inplace=True)
print(f"Rows after dropping nulls: {len(df)}")
 
X = df[FEATURES].values
y = df["risk_score"].values
 
# Scale features (important for some models; doesn't hurt tree models)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
 
# Train/Test Split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]}")
print(f"Test  size: {X_test.shape[0]}")
 
 


STEP 3: Preprocessing
Rows after dropping nulls: 50000
Train size: 40000
Test  size: 10000


In [6]:
# ─────────────────────────────────────────────
# 4. RULE-BASED LABEL FUNCTION
# ─────────────────────────────────────────────
# Thresholds derived from data distribution
# Adjust these if your data suggests different boundaries
LOW_THRESHOLD    = 35.0
MEDIUM_THRESHOLD = 55.0
 
def score_to_label(score):
    """Convert predicted risk_score to risk_label (rule-based)."""
    if score < LOW_THRESHOLD:
        return "Low"
    elif score < MEDIUM_THRESHOLD:
        return "Medium"
    else:
        return "High"
 
 

In [7]:
# ─────────────────────────────────────────────
# 5. MODEL TRAINING & EVALUATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 4: Model Training & Evaluation")
print("=" * 60)
 
# Build model registry
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ),
}
 
if XGBOOST_AVAILABLE:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        n_jobs=-1,
        random_state=42,
        verbosity=0
    )
 
if LIGHTGBM_AVAILABLE:
    models["LightGBM"] = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=63,
        subsample=0.8,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
 
results = {}
 
for name, model in models.items():
    print(f"\n→ Training {name}...")
    model.fit(X_train, y_train)
 
    y_pred = model.predict(X_test)
 
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
 
    # Classification accuracy using rule-based labels
    y_test_labels = [score_to_label(s) for s in y_test]
    y_pred_labels = [score_to_label(s) for s in y_pred]
 
    from sklearn.metrics import accuracy_score
    label_acc = accuracy_score(y_test_labels, y_pred_labels)
 
    results[name] = {
        "model": model,
        "y_pred": y_pred,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Label_Accuracy": label_acc
    }
 
    print(f"   MAE              : {mae:.4f}")
    print(f"   RMSE             : {rmse:.4f}")
    print(f"   R² Score         : {r2:.4f}")
    print(f"   Label Accuracy   : {label_acc * 100:.2f}%")
 
 


STEP 4: Model Training & Evaluation

→ Training Random Forest...
   MAE              : 0.3660
   RMSE             : 0.4766
   R² Score         : 0.9979
   Label Accuracy   : 98.52%

→ Training XGBoost...
   MAE              : 0.2054
   RMSE             : 0.2591
   R² Score         : 0.9994
   Label Accuracy   : 99.10%

→ Training LightGBM...
   MAE              : 0.2160
   RMSE             : 0.2731
   R² Score         : 0.9993
   Label Accuracy   : 99.04%


In [8]:

# ─────────────────────────────────────────────
# 6. COMPARISON & BEST MODEL SELECTION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5: Model Comparison")
print("=" * 60)
 
comparison_df = pd.DataFrame({
    name: {
        "MAE": r["MAE"],
        "RMSE": r["RMSE"],
        "R²": r["R2"],
        "Label Acc %": r["Label_Accuracy"] * 100
    }
    for name, r in results.items()
}).T.round(4)
 
print(comparison_df.to_string())
 
# Pick best model by R²
best_name = max(results, key=lambda n: results[n]["R2"])
best_model = results[best_name]["model"]
print(f"\n🏆 Best Model: {best_name}  (R² = {results[best_name]['R2']:.4f})")
 
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ["MAE", "RMSE", "R2"]
colors  = ["#e74c3c", "#e67e22", "#2ecc71"]
 
for ax, metric, color in zip(axes, metrics, colors):
    vals = [results[n][metric] for n in results]
    ax.bar(list(results.keys()), vals, color=color, alpha=0.85)
    ax.set_title(metric)
    ax.set_ylabel(metric)
 
plt.suptitle("Model Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/model_comparison.png", dpi=150)
plt.close()
print("✅ Saved: outputs/model_comparison.png")
 


STEP 5: Model Comparison
                  MAE    RMSE      R²  Label Acc %
Random Forest  0.3660  0.4766  0.9979        98.52
XGBoost        0.2054  0.2591  0.9994        99.10
LightGBM       0.2160  0.2731  0.9993        99.04

🏆 Best Model: XGBoost  (R² = 0.9994)
✅ Saved: outputs/model_comparison.png


In [9]:

 
# ─────────────────────────────────────────────
# 7. CONFUSION MATRIX FOR LABELS (BEST MODEL)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6: Label Classification Report (Best Model)")
print("=" * 60)
 
y_pred_best = results[best_name]["y_pred"]
y_test_labels = [score_to_label(s) for s in y_test]
y_pred_labels = [score_to_label(s) for s in y_pred_best]
 
print(classification_report(y_test_labels, y_pred_labels, target_names=["High", "Low", "Medium"]))
 
label_order = ["Low", "Medium", "High"]
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=label_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_order)
disp.plot(colorbar=False, cmap="Blues")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig("outputs/confusion_matrix.png", dpi=150)
plt.close()
print("✅ Saved: outputs/confusion_matrix.png")
 


STEP 6: Label Classification Report (Best Model)
              precision    recall  f1-score   support

        High       0.98      0.94      0.96       192
         Low       0.99      0.99      0.99      5360
      Medium       0.99      0.99      0.99      4448

    accuracy                           0.99     10000
   macro avg       0.99      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000

✅ Saved: outputs/confusion_matrix.png


In [10]:

# ─────────────────────────────────────────────
# 8. FEATURE IMPORTANCE (BEST MODEL)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7: Feature Importance")
print("=" * 60)
 
if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    feat_df = pd.DataFrame({
        "Feature": FEATURES,
        "Importance": importances
    }).sort_values("Importance", ascending=True)
 
    print(feat_df.to_string(index=False))
 
    feat_df.plot(
        kind="barh", x="Feature", y="Importance",
        color="steelblue", legend=False,
        figsize=(7, 3)
    )
    plt.title(f"Feature Importance — {best_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig("outputs/feature_importance.png", dpi=150)
    plt.close()
    print("✅ Saved: outputs/feature_importance.png")
 
 


STEP 7: Feature Importance
   Feature  Importance
       lms    0.045080
assignment    0.157925
     marks    0.347623
attendance    0.449373
✅ Saved: outputs/feature_importance.png


In [11]:

# ─────────────────────────────────────────────
# 9. SAVE MODEL & SCALER
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8: Saving Model & Scaler")
print("=" * 60)
 
joblib.dump(best_model, "outputs/risk_score_model.pkl")
joblib.dump(scaler,     "outputs/scaler.pkl")
print("✅ Saved: outputs/risk_score_model.pkl")
print("✅ Saved: outputs/scaler.pkl")
 
# Save thresholds
thresholds = {
    "low_threshold": LOW_THRESHOLD,
    "medium_threshold": MEDIUM_THRESHOLD
}
import json
with open("outputs/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)
print("✅ Saved: outputs/thresholds.json")
 
 


STEP 8: Saving Model & Scaler
✅ Saved: outputs/risk_score_model.pkl
✅ Saved: outputs/scaler.pkl
✅ Saved: outputs/thresholds.json


In [12]:

# ─────────────────────────────────────────────
# 10. INFERENCE FUNCTION — USE THIS IN YOUR APP
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 9: Inference Demo")
print("=" * 60)
 
def predict_student_risk(attendance, marks, assignment, lms,
                          model=best_model, scaler=scaler):
    """
    Predict risk_score and risk_label for a single student.
 
    Parameters:
        attendance  : float  (0–100)
        marks       : float  (0–100)
        assignment  : float  (0–100)
        lms         : float  (0–100)
 
    Returns:
        dict with keys: risk_score, risk_label, top_factors
    """
    features = np.array([[attendance, marks, assignment, lms]])
    features_scaled = scaler.transform(features)
    score = float(model.predict(features_scaled)[0])
    score = max(0.0, min(100.0, score))  # clip to [0, 100]
    label = score_to_label(score)
 
    # Explain top contributing factors (rule-based explanation)
    factors = []
    if attendance < 75:
        factors.append(f"Low attendance ({attendance}%)")
    if marks < 50:
        factors.append(f"Low internal marks ({marks})")
    if assignment < 60:
        factors.append(f"Low assignment completion ({assignment}%)")
    if lms < 40:
        factors.append(f"Inactive on LMS (score: {lms})")
 
    if not factors:
        factors = ["Performance is within acceptable range"]
 
    return {
        "risk_score": round(score, 2),
        "risk_label": label,
        "top_factors": factors
    }
 
 
# Demo predictions
test_students = [
    {"attendance": 40, "marks": 35, "assignment": 20, "lms": 10,  "expected": "High"},
    {"attendance": 60, "marks": 55, "assignment": 50, "lms": 40,  "expected": "Medium"},
    {"attendance": 90, "marks": 85, "assignment": 95, "lms": 80,  "expected": "Low"},
    {"attendance": 84, "marks": 96, "assignment": 34, "lms": 20,  "expected": "Low"},
]
 
print(f"\n{'Student':<10} {'Score':>8} {'Label':>8} {'Expected':>10}  Top Factors")
print("-" * 80)
for i, s in enumerate(test_students, 1):
    result = predict_student_risk(s["attendance"], s["marks"], s["assignment"], s["lms"])
    match = "✅" if result["risk_label"] == s["expected"] else "❌"
    factors_str = "; ".join(result["top_factors"])
    print(f"Student {i}  {result['risk_score']:>8.2f} {result['risk_label']:>8} {s['expected']:>10} {match}  {factors_str}")
 
 
print("\n" + "=" * 60)
print("✅ PIPELINE COMPLETE")
print("=" * 60)
print(f"\nBest model  : {best_name}")
print(f"R² Score    : {results[best_name]['R2']:.4f}")
print(f"Label Acc   : {results[best_name]['Label_Accuracy']*100:.2f}%")
print("\nSaved files in outputs/:")
print("  risk_score_model.pkl  ← load this for predictions")
print("  scaler.pkl            ← use to scale new input")
print("  thresholds.json       ← Low/Medium/High boundaries")


STEP 9: Inference Demo

Student       Score    Label   Expected  Top Factors
--------------------------------------------------------------------------------
Student 1     65.01     High       High ✅  Low attendance (40%); Low internal marks (35); Low assignment completion (20%); Inactive on LMS (score: 10)
Student 2     45.78   Medium     Medium ✅  Low attendance (60%); Low assignment completion (50%)
Student 3     11.31      Low        Low ✅  Performance is within acceptable range
Student 4     28.64      Low        Low ✅  Low assignment completion (34%); Inactive on LMS (score: 20)

✅ PIPELINE COMPLETE

Best model  : XGBoost
R² Score    : 0.9994
Label Acc   : 99.10%

Saved files in outputs/:
  risk_score_model.pkl  ← load this for predictions
  scaler.pkl            ← use to scale new input
  thresholds.json       ← Low/Medium/High boundaries
